# Notebook 7 ? Ubah Bentuk Tabel

Ubah tabel lebar menjadi panjang (wide ke long) atau panjang menjadi lebar (long ke wide). Contohnya, kolom penjualan per bulan dapat diubah menjadi satu kolom bulan dan satu kolom nilai.

In [ ]:
from pathlib import Path
import pandas as pd

# KONFIGURASI
INPUT_PATH = Path("Input/input.csv")
SEP = ","
MODE = "wide_to_long"  # "wide_to_long" atau "long_to_wide"
# Untuk wide_to_long:
ID_COLUMNS = ["id"]
VALUE_COLUMNS = ["jan", "feb", "mar"]
VARIABLE_NAME = "bulan"
VALUE_NAME = "nilai"
# Untuk long_to_wide:
INDEX_COLUMNS = ["id"]
PIVOT_COLUMN = "bulan"
PIVOT_VALUE = "nilai"
AGGREGATION = "first"  # "first", "sum", "mean", "min", "max"
OUTPUT_PATH = Path("Output/hasil_reshape.csv")
def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Ubah bentuk dan simpan

In [ ]:
source = INPUT_PATH if INPUT_PATH.is_absolute() else BASE_DIR / INPUT_PATH
if not source.exists(): raise FileNotFoundError(f"File tidak ditemukan: {source}")
df = pd.read_csv(source, sep=SEP, dtype="string", encoding="utf-8-sig")
if MODE == "wide_to_long":
    needed = ID_COLUMNS + VALUE_COLUMNS
    missing = [c for c in needed if c not in df]
    if missing: raise KeyError(f"Kolom tidak ditemukan: {missing}")
    result = df.melt(id_vars=ID_COLUMNS, value_vars=VALUE_COLUMNS, var_name=VARIABLE_NAME, value_name=VALUE_NAME)
elif MODE == "long_to_wide":
    needed = INDEX_COLUMNS + [PIVOT_COLUMN, PIVOT_VALUE]
    missing = [c for c in needed if c not in df]
    if missing: raise KeyError(f"Kolom tidak ditemukan: {missing}")
    if AGGREGATION not in {"first", "sum", "mean", "min", "max"}: raise ValueError("AGGREGATION tidak didukung.")
    if AGGREGATION in {"sum", "mean"}: df[PIVOT_VALUE] = pd.to_numeric(df[PIVOT_VALUE], errors="coerce")
    result = df.pivot_table(index=INDEX_COLUMNS, columns=PIVOT_COLUMN, values=PIVOT_VALUE, aggfunc=AGGREGATION).reset_index()
    result.columns.name = None
else:
    raise ValueError("MODE harus 'wide_to_long' atau 'long_to_wide'.")

display(result.head())
output = OUTPUT_PATH if OUTPUT_PATH.is_absolute() else BASE_DIR / OUTPUT_PATH
output.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output, index=False, encoding="utf-8-sig")
print(f"Tersimpan: {output.resolve()} | {len(result):,} baris x {len(result.columns)} kolom")
